# Piloto FINRISK con hiperparámetros optimizados

Ejecuta los tres modelos y registra cuatro corridas nuevas con el prefijo **Piloto optimizado**. Conserva el notebook y los resultados originales. La búsqueda de 63 configuraciones está documentada en `results/optimizacion_hiperparametros/`.

La validación de origen 2023 ya se utilizó para seleccionar hiperparámetros. La prueba de origen 2024 es un período previamente consultado; no se utiliza para seguir ajustando. El objetivo es presencia en inventario anual, no una probabilidad calibrada de quiebra.

In [ ]:
from pathlib import Path
import json
import os
import pandas as pd

candidatas = [Path.cwd(), *Path.cwd().parents]
RAIZ = next((p for p in candidatas if (p / "requirements.txt").is_file() and (p / "src").is_dir()), None)
if RAIZ is None:
    raise FileNotFoundError("Abra el notebook desde FINRISK_COLOMBIA o su carpeta notebooks. Revise docs/entrega2/preparacion_datos.md.")
RAIZ_DATOS = Path(os.environ.get("FINRISK_DATOS_RAIZ", str(RAIZ))).expanduser().resolve()
REPORTES = RAIZ_DATOS / "reports/preparacion"
requeridos = [REPORTES / "adquisicion_financieros.json", REPORTES / "auditoria_insolvencia.json", REPORTES / "eventos_observados_insolvencia.csv", RAIZ_DATOS / "data/processed/recuperacion/base_financiera_recuperada_sin_etiquetas.csv.gz"]
faltantes = [p.relative_to(RAIZ_DATOS).as_posix() for p in requeridos if not p.is_file()]
if faltantes:
    raise FileNotFoundError("Faltan archivos de preparación. Ejecute descarga financiera, descarga de insolvencia, auditoría y consolidación. Siga docs/entrega2/preparacion_datos.md. Archivos faltantes: " + ", ".join(faltantes))
pd.set_option("display.max_columns", 12)

import sys
import subprocess
from IPython.display import display, Image
sys.path.insert(0, str(RAIZ / "src"))


## Parámetros seleccionados y fuentes locales

In [ ]:
from finrisk_piloto_datos import cargar_construir
from finrisk_piloto_modelos import entrenar_piloto, FEATURES, _modelos

seleccion = json.loads((RAIZ / "results/optimizacion_hiperparametros/mejores_hiperparametros.json").read_text(encoding="utf-8"))
datos, protocolo, cohortes = cargar_construir(RAIZ_DATOS)
if protocolo["sha256_dataset"] != seleccion["sha256_dataset"]:
    raise ValueError("La extracción local difiere de la usada para optimizar.")
for nombre, registro in seleccion["mejores"].items():
    actuales = _modelos()[nombre].get_params()
    assert all(actuales[k] == v for k, v in registro["parametros_elegidos"].items())
display(pd.DataFrame(seleccion["mejores"]).T)
SALIDA_MODELOS = RAIZ_DATOS / ".cache/finrisk/piloto_optimizado"
RESULTADOS = RAIZ / "results/piloto_optimizado"
RESULTADOS.mkdir(parents=True, exist_ok=True)

## Particiones y restricciones temporales

| Uso | Origen | Ejercicio financiero | Inventario objetivo |
|---|---|---|---|
| Entrenamiento | 2021 y 2022 | 2020 y 2021 | diciembre de 2022 y 2023 |
| Validación | 2023 | 2022 | diciembre de 2024 |
| Prueba reservada | 2024 | 2023 | diciembre de 2025 |

La empresa puede repetirse en años distintos. El resultado corresponde a reportantes históricos y no mide exclusivamente empresas nuevas. La separación cronológica no prueba que los reportes y etiquetas estuvieran publicados en cada origen: no acredita un backtest operativo libre de información posterior.

In [ ]:
display(datos.groupby("particion", sort=False).agg(
    observaciones=("evento", "size"), empresas=("nit", "nunique"),
    presencias=("evento", "sum"), prevalencia=("evento", "mean"),
))
display(pd.Series(protocolo["intersecciones_nit_particiones"],
                  name="empresas_compartidas_entre_particiones"))
display(cohortes[["anio_origen", "n_financieros",
    "excluidos_antecedente_observado", "excluidos_coherencia",
    "excluidos_sin_indicadores", "n_modelado",
    "presencias_excluidos_coherencia", "con_indicadores_imputables"]])


## Tres modelos y selección previa a prueba

Se comparan referencia constante de prevalencia, regresión logística y Random Forest. Hiperparámetros y semilla están en src/finrisk_piloto_modelos.py; no se buscaron configuraciones con prueba. Imputación y escalado se ajustan solo con entrenamiento. La regresión aplica arcsinh para conservar negativos.

Se elige el modelo por mayor **AP (average precision)** en validación y el umbral por máximo **F2** en validación. F2 es un criterio técnico que pondera sensibilidad; no representa costos ni capacidad de revisión del banco. En empates se elige el modelo más simple según orden predefinido y el umbral menor con tolerancia 1e-12.

La selección y los hashes se sellan antes de evaluar una vez al ganador en prueba, sin reajustar con validación. Una ejecución íntegra se recupera de caché sin reevaluar prueba.

In [ ]:
resumen = entrenar_piloto(datos, protocolo, SALIDA_MODELOS)
print("Resultado recuperado de caché:", resumen["resultado_en_cache"])
comparacion = pd.DataFrame(resumen["modelos"])
columnas = ["modelo", "ap", "sensibilidad", "precision_alertas", "f2",
            "umbral", "alertas", "fraccion_alertas"]
display(comparacion[columnas].set_index("modelo"))
print("Ganador por AP de validación:", resumen["ganador"])
print("Umbral técnico sellado:", resumen["umbral"])
print("SHA de selección:", resumen["sha256_seleccion"])


### Comparación visual en validación

AP es el promedio de precisión ponderado por incrementos de sensibilidad; no representa la precisión al umbral ni una probabilidad individual. La referencia constante tiene AP igual a la prevalencia de validación. Mejorar esa referencia no garantiza precisión operativa suficiente.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from sklearn.metrics import precision_recall_curve
FIGURAS = RESULTADOS / "figuras"
FIGURAS.mkdir(parents=True, exist_ok=True)
etiquetas = {
    "referencia_prevalencia": "Referencia de prevalencia",
    "regresion_logistica": "Regresión logística",
    "bosque_aleatorio": "Random Forest",
}
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), layout="constrained")
colores = ["#8b95a5", "#d48a24", "#177b8e"]
axes[0].barh([etiquetas[x] for x in comparacion.modelo], comparacion.ap, color=colores)
axes[0].set(xlabel="AP en validación (escala 0–1)", title="Selección por AP")
axes[0].set_xlim(0, max(comparacion.ap) * 1.25)
for i, valor in enumerate(comparacion.ap):
    axes[0].text(valor + .00025, i, f"{valor:.4f}", va="center")
for registro, color in zip(resumen["modelos"], colores):
    # round_trip conserva exactamente el puntaje de la referencia constante.
    pred = pd.read_csv(SALIDA_MODELOS / registro["archivo_predicciones"],
                       float_precision="round_trip")
    if registro["modelo"] == "referencia_prevalencia":
        axes[1].axhline(registro["prevalencia"], color=color, linestyle="--",
                        label="Prevalencia de validación", linewidth=1.5)
        continue
    precision, sensibilidad, _ = precision_recall_curve(pred["evento"], pred["puntaje"])
    axes[1].plot(sensibilidad, precision, color=color,
                 label=etiquetas[registro["modelo"]], linewidth=1.5)
axes[1].set(xlim=(0, 1), ylim=(0, .12), xlabel="Sensibilidad",
            ylabel="Precisión de alertas", title="Validación (detalle: precisión ≤12%)")
axes[1].xaxis.set_major_formatter(PercentFormatter(1))
axes[1].yaxis.set_major_formatter(PercentFormatter(1))
axes[1].legend(fontsize=8)
fig.savefig(FIGURAS / "comparacion_validacion.png", dpi=160)
plt.close(fig)
display(Image(filename=str(FIGURAS / "comparacion_validacion.png")))


In [ ]:
for registro in resumen["modelos"]:
    if registro["modelo"] in seleccion["mejores"]:
        esperado = seleccion["mejores"][registro["modelo"]]["ap_elegida"]
        assert abs(registro["ap"] - esperado) < 1e-12, "AP distinta de la búsqueda"
print("AP de validación reproducidas.")

## Prueba del modelo seleccionado

Se aplica el umbral sellado sin reajustar. El período ya fue consultado en el piloto original; esta comparación no constituye una prueba nueva independiente.

In [ ]:
prueba = resumen["prueba"]
display(pd.Series({k: prueba[k] for k in [
    "modelo", "n", "positivos", "prevalencia", "ap", "umbral", "sensibilidad",
    "precision_alertas", "f2", "alertas", "fraccion_alertas",
    "numero_evaluaciones_prueba", "modelo_reajustado_antes_prueba"
]}, name="prueba_reservada"))
mc = prueba["matriz_confusion"]
matriz = [[mc["vn"], mc["fp"]], [mc["fn"], mc["vp"]]]
fig, ax = plt.subplots(figsize=(6.6, 4.4), layout="constrained")
ax.imshow(matriz, cmap="Blues")
for i in range(2):
    for j in range(2):
        ax.text(j, i, f"{matriz[i][j]:,}", ha="center", va="center",
                color="white" if (i, j) == (0, 0) else "#13293d", fontsize=15)
ax.set(xticks=[0, 1], xticklabels=["Sin alerta", "Con alerta"],
       yticks=[0, 1], yticklabels=["Ausencia en inventario", "Presencia en inventario"],
       xlabel="Salida con el umbral sellado", ylabel="Registro observado",
       title="Prueba · origen 2024 → inventario 2025")
fig.savefig(FIGURAS / "matriz_prueba.png", dpi=160)
plt.close(fig)
display(Image(filename=str(FIGURAS / "matriz_prueba.png")))


## Resultados agregados

Se exportan métricas y huellas; los modelos y predicciones individuales permanecen en la caché local. Los puntajes no están calibrados.

In [ ]:
import shutil
for nombre in ["protocolo_etiqueta.json", "configuracion_modelado.json", "firma_ejecucion.json", "seleccion_sellada.json", "resumen_piloto.json"]:
    shutil.copyfile(SALIDA_MODELOS / nombre, RESULTADOS / nombre)
for archivo in (SALIDA_MODELOS / "agregados").iterdir():
    if archivo.is_file():
        shutil.copyfile(archivo, RESULTADOS / archivo.name)
cohortes.to_csv(RESULTADOS / "cohortes_piloto.csv", index=False)

## Registro y recuperación en MLflow

Experimento Proyecto-MIAD. Se registran tres validaciones y una prueba con el prefijo Piloto optimizado. El registro se activa explícitamente mediante FINRISK_REGISTRAR_MLFLOW=1. Se comprueba que los scripts y este notebook coincidan con el commit local antes de publicar. No se suben filas individuales.

In [ ]:
REGISTRAR_MLFLOW = os.environ.get("FINRISK_REGISTRAR_MLFLOW", "0") == "1"
COMMIT_CODIGO = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=RAIZ, text=True).strip()
MANIFIESTO = RESULTADOS / "registro_mlflow.json"
if REGISTRAR_MLFLOW:
    archivos = ["src/finrisk_piloto_modelos.py", "src/finrisk_piloto_datos.py",
                "src/finrisk_piloto_mlflow.py", "src/finrisk_piloto_optimizacion.py",
                "notebooks/03_Optimizacion_Modelos_FINRISK.ipynb",
                "results/optimizacion_hiperparametros/mejores_hiperparametros.json"]
    for archivo in archivos:
        versionado = subprocess.check_output(["git", "show", f"{COMMIT_CODIGO}:{archivo}"], cwd=RAIZ)
        if versionado != (RAIZ / archivo).read_bytes():
            raise ValueError(f"Cambios sin incluir en el commit: {archivo}")
    from finrisk_piloto_mlflow import registrar_piloto
    registro = registrar_piloto(
        SALIDA_MODELOS / "resumen_piloto.json", commit_git=COMMIT_CODIGO,
        salida=MANIFIESTO, prefijo_run="Piloto optimizado")
if MANIFIESTO.is_file():
    registro = json.loads(MANIFIESTO.read_text(encoding="utf-8"))
    display(pd.DataFrame(registro["corridas_validacion"] + [registro["corrida_prueba"]]))
    display(pd.Series({"commit": registro["commit_fuente"],
        "pipeline_recuperado_identico": registro["pipeline_recuperado_identico"],
        "sha256": registro["pipeline_recuperado_sha256"]}))
else:
    print("Registro desactivado. Activar FINRISK_REGISTRAR_MLFLOW=1 para registrar.")